In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Load data
data = pd.read_csv("commercial-banks/ADBL.csv")
prices = data[['close']].values

# Scale
scaler = MinMaxScaler()
prices_scaled = scaler.fit_transform(prices)

# Create sequences
X, y = [], []
seq_len = 30
for i in range(seq_len, len(prices_scaled)):
    X.append(prices_scaled[i-seq_len:i, 0])
    y.append(prices_scaled[i, 0])

X, y = np.array(X), np.array(y)
X = X.reshape((X.shape[0], X.shape[1], 1))

# Train-test split
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# LSTM model
model = Sequential([
    LSTM(50, input_shape=(X_train.shape[1], 1)),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

# Prediction
pred = model.predict(X_test)
pred = scaler.inverse_transform(pred)
actual = scaler.inverse_transform(y_test.reshape(-1,1))

# Metrics
rmse = np.sqrt(mean_squared_error(actual, pred))
mae = mean_absolute_error(actual, pred)

print("LSTM RMSE:", rmse)
print("LSTM MAE:", mae)


/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
LSTM RMSE: 8.349186799272477
LSTM MAE: 5.939061587555237


In [5]:
data

,published_date,open,high,low,close,per_change,traded_quantity,traded_amount,status
0,2010-09-16,117.0,122.0,116.0,120.0,NaN,5280.0,0.0,0
1,2010-09-19,120.0,120.0,116.0,118.0,-1.67,2648.0,0.0,0
2,2010-09-20,118.0,119.0,116.0,118.0,0.00,2346.0,0.0,0
3,2010-09-21,118.0,118.0,115.0,116.0,-1.69,7160.0,0.0,0
4,2010-09-23,116.0,120.0,117.0,120.0,3.45,8417.0,0.0,0
...,...,...,...,...,...,...,...,...,...
3487,2025-12-22,288.0,290.0,286.0,288.8,-0.35,20208.0,5801029.3,1
3488,2025-12-23,288.0,290.0,285.2,286.9,-0.66,23575.0,6746825.5,-1
3489,2025-12-24,286.0,290.0,285.2,288.0,0.38,20792.0,5948636.7,1
3490,2025-12-28,291.0,316.8,286.0,297.0,3.13,49373.0,14481034.8,1
